In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data = fetch_california_housing(as_frame=True)
df = data.frame        # ✅ correct — includes target
df.head()
df.isnull().sum()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


In [2]:
X=df.drop(columns='MedHouseVal')
y=df['MedHouseVal']


In [3]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
#handle missing values and scale the data
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), X_train.columns),
        ('simple_imputer', SimpleImputer(strategy='mean'), X_train.columns)
    ]
)
# now train the model by RandomForestRegressor

model = RandomForestRegressor(random_state=42,max_depth=20,n_estimators=200)
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('model', model)])
model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)
# Evaluate the model
from sklearn.metrics import mean_squared_error, r2_score
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f'Mean Squared Error: {mse}')
print(f'R-squared: {r2}')



Mean Squared Error: 0.25472679029150364
R-squared: 0.8056127556196613


In [11]:
#now apply KFold cross validation to evaluate the model
from sklearn.model_selection import cross_val_score, KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model_pipeline, X_train, y_train, cv=kf, scoring='r2')
print(f'Cross-validation R-squared scores: {cv_scores}')
print(f'Mean R-squared score: {cv_scores.mean()}')

Cross-validation R-squared scores: [0.79908443 0.8132617  0.80887404 0.79614416 0.80539715]
Mean R-squared score: 0.8045522953694135


In [6]:
import json
from pathlib import Path

ARTIFACT_DIR = Path.cwd().parent / "backend" / "artifacts"  # same folder as model.pkl
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Feature importance from the trained model
importances = model_pipeline.named_steps["model"].feature_importances_
feat_importance = dict(zip(X.columns, importances.tolist()))
feat_importance = dict(sorted(feat_importance.items(), key=lambda x: -x[1]))

meta = {
    "model_type": "RandomForestRegressor",
    "n_estimators": model_pipeline.named_steps["model"].n_estimators,
    "max_depth": model_pipeline.named_steps["model"].max_depth,
    "features": list(X.columns),
    "feature_importance": feat_importance,
    "metrics": {"mse": mse, "r2": r2},
    "target_description": "Median house value in units of $100,000 (California districts)",
    "dataset_summary": X.describe().to_dict(),
    "correlations_with_target": df.corr()["MedHouseVal"].drop("MedHouseVal").to_dict()
}

with open(ARTIFACT_DIR / "model_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved to:", (ARTIFACT_DIR / "model_meta.json").resolve())
print("Size:", (ARTIFACT_DIR / "model_meta.json").stat().st_size, "bytes")

Saved to: D:\Project2\backend\artifacts\model_meta.json
Size: 3164 bytes


In [5]:
from pathlib import Path
import pickle

PROJECT_ROOT = Path("D:/Project2")
ARTIFACT_DIR = PROJECT_ROOT / "backend" / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

target_path = ARTIFACT_DIR / "model.pkl"
print("Writing to:", target_path.resolve())

with open(target_path, "wb") as f:
    pickle.dump(model_pipeline, f)

print("File exists after write:", target_path.exists())
print("Size:", target_path.stat().st_size, "bytes")

Writing to: D:\Project2\backend\artifacts\model.pkl
File exists after write: True
Size: 238108168 bytes


In [7]:
df.to_csv(ARTIFACT_DIR / "california_housing.csv", index=False)
print("Saved CSV:", (ARTIFACT_DIR / "california_housing.csv").exists())

Saved CSV: True
